# Python Bridge for 10a

## A Short Code Refresher

Use this optional notebook alongside 10a to practice turning fixed anomaly scores into flags and comparing those flags with a separate outcome. Allow about 15 minutes. Eight invented inspection records make every count visible without rerunning the full wine analysis.

You will use Boolean comparisons, masks, counts, guarded division, and a short helper function. The scores and outcomes stay fixed while the threshold changes.

## References

The pandas [indexing guide](https://pandas.pydata.org/docs/user_guide/indexing.html#boolean-indexing) explains Boolean selection. Scikit-learn's [precision and recall discussion](https://scikit-learn.org/stable/modules/model_evaluation.html#precision-recall-f-measure-metrics) supplies the definitions used in 10a. We calculate the counts directly here so the denominators remain visible.

## Keep Scores and Outcomes Separate

The score column has already been oriented so larger means more unusual. `Fault found` records a hypothetical independent inspection result for every item, including those we will not flag. These are supplied teaching values, not fitted detector output or real equipment records.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

records = pd.DataFrame(
    {
        "Score": [0.90, 0.80, 0.80, 0.60, 0.40, 0.30, 0.20, 0.10],
        "Fault found": [True, False, True, False, True, False, False, False],
    },
    index=list("ABCDEFGH"),
)
display(records)

The outcome was not used to calculate the score or choose the following illustrative threshold. Having outcomes for every record lets us count missed faults; a real investigation of flagged records alone would not provide that information.

## Make and Apply a Boolean Mask

Use `.ge()` for “greater than or equal to.” The returned Series contains one Boolean per record with the same index. Pass it to `.loc` to inspect flagged rows.

In [ ]:
threshold = 0.80
selected = records["Score"].ge(threshold)
fault = records["Fault found"]

display(records.assign(Flagged=selected))
print("Review count:", int(selected.sum()))
display(records.loc[selected])

A, B, and C are flagged, giving three reviews. Boolean sums count `True` values. The equality in the rule includes both records scoring 0.80.

## Count the Four Outcomes

`&` requires both conditions and `~` reverses a Boolean mask. Use these elementwise operators for Series. Python's `and` asks for one truth value and is unsuitable for a whole column.

In [ ]:
tp = int((selected & fault).sum())
fp = int((selected & ~fault).sum())
fn = int((~selected & fault).sum())
tn = int((~selected & ~fault).sum())
confusion = pd.DataFrame(
    [[tp, fp], [fn, tn]],
    index=["Flagged", "Unflagged"],
    columns=["Fault found", "No fault found"],
)
display(confusion)
print("Counted records:", tp + fp + fn + tn)

The table contains two true positives, one false positive, one false negative, and four true negatives. The row totals describe our decisions; the column totals describe the inspection outcomes. Moving the threshold can change a record's decision, but not whether its inspection found a fault.

## Calculate Precision and Recall

Precision divides the faults found among flagged records by the review count. Recall divides the faults found among flagged records by all observed faults. Return `np.nan` when a denominator is zero so an undefined ratio does not appear as a measured zero.

In [ ]:
precision = tp / (tp + fp) if tp + fp else np.nan
recall = tp / (tp + fn) if tp + fn else np.nan
display(pd.Series({"Precision": precision, "Recall": recall}))

Both happen to equal two-thirds here, but their denominators answer different questions. Selecting no records would give undefined precision and zero recall because this table contains three faults.

## Reuse the Calculation at Several Thresholds

Put the counting steps in a function that returns a labeled Series. A dictionary comprehension calls it at four cutoffs; transposing the resulting DataFrame puts one threshold on each row.

In [ ]:
def summarize_threshold(cutoff):
    flags = records["Score"].ge(cutoff)
    hits = int((flags & fault).sum())
    reviews = int(flags.sum())
    total_faults = int(fault.sum())
    return pd.Series(
        {
            "Reviews": reviews,
            "Faults found": hits,
            "Precision": hits / reviews if reviews else np.nan,
            "Recall": hits / total_faults if total_faults else np.nan,
        }
    )


summary = pd.DataFrame({cutoff: summarize_threshold(cutoff) for cutoff in [1.0, 0.8, 0.4, 0.0]}).T
summary.index.name = "Threshold"
display(summary)

Lowering the threshold from 0.80 to 0.40 adds D and E. Only E has a fault, so recall rises to 1 while precision falls to 3/5. Selecting everything gives precision 3/8. Precision depends on the outcomes entering the list; it need not always decrease as the threshold falls.

Plot the scores and mark the cutoff. The outcome markers let you check the confusion counts against individual records.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5), layout="constrained")
for outcome, marker in [(True, "X"), (False, "o")]:
    subset = records.loc[records["Fault found"].eq(outcome)]
    positions = records.index.get_indexer(subset.index)
    ax.scatter(positions, subset["Score"], marker=marker, label=f"Fault found: {outcome}")
ax.set_xticks(range(len(records)), records.index)
ax.axhline(threshold, color="black", linestyle="--", label="Threshold 0.80")
ax.set(xlabel="Record", ylabel="Oriented anomaly score", title="Fixed Scores, Chosen Cutoff")
ax.legend()
plt.show()

## Try One Small Modification

Request the second-highest score and use it as a threshold. Predict the review count before running this cell. Does a budget of two produce exactly two flags?

In [ ]:
budget = 2
budget_threshold = records["Score"].nlargest(budget).iloc[-1]
print("Boundary score:", budget_threshold)
display(summarize_threshold(budget_threshold))

The boundary is 0.80 and all three records at or above it are included. This threshold rule includes ties. An exact two-record allocation would require an additional tie-breaking rule.

## Ready for 10a

Return to the lecture when you can build and inspect a flag mask, identify all four outcome counts, explain each metric's denominator, and recognize how ties affect workload. Use a separate evaluation set for a final performance claim after choosing detector settings or thresholds using outcomes.

---

Auburn University / Industrial and Systems Engineering<br>
INSY 7130, Pattern Discovery and Time Series Analysis<br>
© Copyright Danny J. O'Leary.

For course materials, attribution, and licensing information, see the [INSY 7130 course-materials README](../../README.md).